# 04 — Baseline Evaluation (Deliverable 2)

Establishes a quantitative baseline for FitRAG against the gold evaluation set in `data/eval/qa_set.json` (30 Qs: 20 in_scope / 5 ambiguous / 5 out_of_scope).

All evaluation logic lives in **`src/fitrag_eval.py`** — every experiment is one call to `evaluate(config)`, so results are comparable and reproducible (directly addresses the D1 feedback: *quantitative evidence, systematic testing*).

**Metrics**
- *Retrieval (objective, uses `expected_sources`):* Recall@k, Precision@k, MRR, Hit-rate
- *Answer quality (LLM-as-judge = `gpt-oss-120b`, a different model family from the llama generator → reduces self-evaluation bias):* Correctness, Faithfulness
- *Robustness:* Refusal accuracy (out_of_scope → refuse; in_scope → don't refuse)

> **Judge note:** the judge was originally planned as Gemini, but Google's free tier caps at **20 requests/day** — far too few for the experiment matrix. We use a *different-family* Groq model (`gpt-oss-120b`) as judge instead, which preserves the bias-avoidance rationale while running within Groq's limits.

In [ ]:
import sys, json
from pathlib import Path

# Make the repo root importable so `from src import fitrag_eval` works from notebooks/
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from . import eval as fe

print('✅ Harness imported')
print('   Repo root      :', REPO_ROOT)
print('   Gold set       :', fe.GOLD_SET_PATH.relative_to(REPO_ROOT))
print('   Results dir    :', fe.RESULTS_DIR.relative_to(REPO_ROOT))

✅ Harness imported
   Repo root      : c:\Users\livex\Desktop\Uni\FitRAG
   Gold set       : data\eval\qa_set.json
   Results dir    : data\eval\results


## Baseline configuration

This mirrors the current pipeline exactly (notebooks 01–03): MMR retrieval (k=5, fetch_k=20, λ=0.5) over the `multi-qa-MiniLM-L6` FAISS index, strict-grounded prompt, Groq `llama-3.3-70b-versatile` generator. Judge = Groq `gpt-oss-120b` (different model family).

In [2]:
print(json.dumps(fe.BASELINE_CONFIG, indent=2))

{
  "name": "baseline",
  "embedding_model": "multi-qa-MiniLM-L6-cos-v1",
  "index_path": "embeddings/vector_store",
  "search_type": "mmr",
  "k": 5,
  "fetch_k": 20,
  "lambda_mult": 0.5,
  "use_retrieval": true,
  "llm_backend": "groq",
  "llm_model": "llama-3.3-70b-versatile",
  "temperature": 0,
  "max_tokens": 1024,
  "prompt_variant": "strict",
  "judge": true,
  "judge_backend": "groq",
  "judge_model": "openai/gpt-oss-120b",
  "judge_sleep": 0.3
}


## Run the baseline evaluation

~30 generation calls + up to ~50 judge calls, all on Groq. `judge_sleep` paces the calls; `_invoke_with_retry` backs off on rate-limit (429) errors.

In [3]:
# Free-tier note: the Groq generator (llama-3.3-70b) is capped at 100k tokens/day and
# the judge has its own daily cap. To keep this notebook re-runnable, we cache the full
# result to data/eval/results/baseline.json. The generator runs at temperature=0, so
# reusing saved answers is reproducible.
#   - If baseline.json exists  -> load it (free, deterministic).
#   - To recompute from scratch -> delete baseline.json and re-run this cell.
baseline_path = fe.RESULTS_DIR / 'baseline.json'
if baseline_path.exists():
    baseline = json.load(open(baseline_path, encoding='utf-8'))
    print(f'Loaded cached baseline results from {baseline_path.name} '
          f'({baseline["aggregate"]["n_questions"]} questions).')
    print('Delete this file and re-run to recompute live via evaluate().')
else:
    baseline = fe.evaluate({'name': 'baseline'})

Loaded cached baseline results from baseline.json (30 questions).
Delete this file and re-run to recompute live via evaluate().


## Aggregate metrics

In [4]:
agg = baseline['aggregate']
summary = pd.DataFrame(
    [(k, round(v, 3) if isinstance(v, float) else v) for k, v in agg.items()],
    columns=['metric', 'value']
)
summary

,metric,value
0,n_questions,30.000
1,recall_at_k,0.868
2,precision_at_k,0.575
3,mrr,0.771
4,hit_rate,0.917
5,recall_at_k_in_scope,0.925
6,correctness_in_scope,0.588
7,correctness_ambiguous,0.500
8,faithfulness,0.978
9,refusal_accuracy,0.960


## Per-question detail

Full table — retrieval hit/recall, judge scores, and refusal flag per question. This is the evidence base for the failure analysis and hypothesis write-ups in 05 and 06.

In [5]:
rows = []
for r in baseline['per_question']:
    rows.append({
        'id': r['id'],
        'category': r['category'],
        'recall@k': r.get('recall_at_k'),
        'prec@k': r.get('precision_at_k'),
        'RR': r.get('reciprocal_rank'),
        'correct': r.get('correctness'),
        'faith': r.get('faithfulness'),
        'refused': r['refused'],
    })
detail = pd.DataFrame(rows).set_index('id')
detail.round(3)

,category,recall@k,prec@k,RR,correct,faith,refused
id,,,,,,,
Q01,in_scope,1.000,1.0,1.00,1.00,1.0,False
Q02,in_scope,1.000,0.8,1.00,1.00,1.0,False
Q03,in_scope,1.000,1.0,1.00,0.50,1.0,False
Q04,in_scope,1.000,0.2,1.00,1.00,1.0,False
Q05,in_scope,1.000,0.4,1.00,0.75,1.0,False
Q06,in_scope,0.000,0.0,0.00,1.00,1.0,False
Q07,in_scope,1.000,0.6,1.00,0.50,1.0,False
Q08,in_scope,1.000,0.8,0.50,0.50,1.0,False
Q09,in_scope,0.500,0.6,1.00,0.50,1.0,False


## Failure surface

Quick scan for systematic problems — each category is investigated in depth in `06_analysis.ipynb`:
- **Retrieval miss** — in_scope/ambiguous question where no expected source was retrieved (hit = 0).
- **Wrong refusal** — in_scope question that was refused (should have answered).
- **Missed refusal** — out_of_scope question that was answered (should have refused).
- **Low faithfulness** — answered question with faithfulness < 0.75 (possible hallucination).

In [6]:
def show_failures(result):
    for r in result['per_question']:
        problems = []
        if r.get('expected_sources') and r.get('hit') == 0.0:
            problems.append('RETRIEVAL_MISS')
        if r['category'] == 'in_scope' and r['refused']:
            problems.append('WRONG_REFUSAL')
        if r['category'] == 'out_of_scope' and not r['refused']:
            problems.append('MISSED_REFUSAL')
        if r.get('faithfulness') is not None and r['faithfulness'] < 0.75:
            problems.append('LOW_FAITHFULNESS')
        if problems:
            print(f"{r['id']} [{r['category']}] -> {', '.join(problems)}")
            print(f"   Q: {r['question']}")
            print(f"   expected : {r['expected_sources']}")
            print(f"   retrieved: {r['retrieved_sources']}")
            if r.get('correctness_reason'):
                print(f"   judge    : {r['correctness_reason']}")
            print()

show_failures(baseline)

Q06 [in_scope] -> RETRIEVAL_MISS
   Q: Is resistance training safe for children and adolescents according to the NSCA youth position statement?
   expected : ['NSCA_2.pdf']
   retrieved: ['NSCA_5.pdf', 'NSCA_4.pdf', 'NSCA_5.pdf', 'NSCA_5.pdf', 'NSCA_5.pdf']
   judge    : Accurately states that properly designed, supervised resistance training is safe for youth, matching the reference.

Q11 [in_scope] -> LOW_FAITHFULNESS
   Q: Does the WHO recommend any physical activity even for people who cannot meet the full guidelines?
   expected : ['WHO.pdf']
   retrieved: ['WHO.pdf', 'WHO.pdf', 'WHO.pdf', 'WHO.pdf', 'WHO.pdf']
   judge    : Accurately states that any activity is better than none and advises acting within abilities, matching the reference.

Q20 [in_scope] -> WRONG_REFUSAL
   Q: What recovery considerations does the NSCA LTAD position statement emphasize?
   expected : ['NSCA_4.pdf']
   retrieved: ['NSCA_2.pdf', 'NSCA_1.pdf', 'NSCA_3.pdf', 'NSCA_4.pdf', 'NSCA_1.pdf']
   judge    : 